# Action Label Dataset Inspector

This notebook inspects recorded transition data from `sample_retro_transitions.py` and focuses on **ground-truth action labels** per game run.

In [13]:
from pathlib import Path
import json
from collections import Counter, defaultdict

DATA_PATH = Path('data/retro_experiment_clean_smoke_5games/transitions.jsonl')
assert DATA_PATH.exists(), f'Missing file: {DATA_PATH}'

records = []
with DATA_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f'Loaded {len(records)} transitions from {DATA_PATH}')
print('Sample keys:', sorted(records[0].keys()))

Loaded 1000 transitions from data/retro_experiment_clean_smoke_5games/transitions.jsonl
Sample keys: ['action_label', 'action_vector', 'episode', 'frame_t', 'frame_tp1', 'game', 'pressed_buttons', 'reward', 'step', 'system', 'terminated', 'truncated']


In [17]:
# Per-game counts and label diversity
per_game_count = Counter(r['game'] for r in records)
per_game_unique_labels = defaultdict(set)
for r in records:
    per_game_unique_labels[r['game']].add(r['action_label'])

print('Per-game transition counts and unique action labels:')
for game in sorted(per_game_count):
    print(f'- {game}: transitions={per_game_count[game]}, unique_labels={len(per_game_unique_labels[game])}')

Per-game transition counts and unique action labels:
- 1942-Nes-v0: transitions=200, unique_labels=25
- AddamsFamily-Sms-v0: transitions=200, unique_labels=20
- AdventureIslandII-Nes-v0: transitions=200, unique_labels=25
- NinjaGaiden-Nes-v0: transitions=200, unique_labels=25
- SonicTheHedgehog-Sms-v0: transitions=200, unique_labels=20


In [18]:
# Top action labels for each game
top_k = 15
label_counts_by_game = defaultdict(Counter)
for r in records:
    label_counts_by_game[r['game']][r['action_label']] += 1

for game in sorted(label_counts_by_game):
    print(f'\n=== {game} (top {top_k} labels) ===')
    for label, count in label_counts_by_game[game].most_common(top_k):
        print(f'{count:4d}  {label}')


=== 1942-Nes-v0 (top 15 labels) ===
  13  NOOP
  13  RIGHT
  12  A+UP
  12  LEFT
  11  B+DOWN
  10  B+RIGHT
  10  B+UP
   9  A+DOWN
   9  LEFT+SELECT
   9  DOWN
   8  A
   8  A+LEFT
   8  A+RIGHT
   7  LEFT+START
   7  START

=== AddamsFamily-Sms-v0 (top 15 labels) ===
  14  UP
  14  PAUSE+UP
  14  DOWN+PAUSE
  12  B+UP
  12  B+RIGHT
  12  A+LEFT
  12  A+UP
  11  A+RIGHT
  11  B+DOWN
  10  B+LEFT
   9  PAUSE+RIGHT
   9  A+DOWN
   9  RIGHT
   8  A
   8  NOOP

=== AdventureIslandII-Nes-v0 (top 15 labels) ===
  20  SELECT+UP
  17  UP
  12  RIGHT
  12  A+RIGHT
  10  A
  10  LEFT+START
   8  B+LEFT
   8  A+LEFT
   8  A+UP
   8  DOWN+SELECT
   8  DOWN+START
   8  RIGHT+START
   7  B
   7  START
   7  B+DOWN

=== NinjaGaiden-Nes-v0 (top 15 labels) ===
  16  SELECT
  14  A+UP
  12  START+UP
  11  LEFT+SELECT
  10  A+LEFT
  10  NOOP
  10  RIGHT+SELECT
   9  DOWN+SELECT
   9  DOWN
   9  A+DOWN
   8  B
   8  START
   7  B+RIGHT
   7  LEFT
   7  B+DOWN

=== SonicTheHedgehog-Sms-v0 (top 15 labels)

In [16]:
# Inspect one game run step-by-step
GAME = sorted(set(r['game'] for r in records))[0]
EPISODE = 0
N = 40

subset = [
    r for r in records
    if r['game'] == GAME and int(r['episode']) == EPISODE
]
subset = sorted(subset, key=lambda x: int(x['step']))

print(f'Inspecting GAME={GAME}, EPISODE={EPISODE}, rows={len(subset)}')
print('step | action_label | pressed_buttons')
print('-' * 100)
for r in subset[:N]:
    step = int(r['step'])
    label = r['action_label']
    pressed = ','.join(r['pressed_buttons'])
    print(f'{step:4d} | {label:35s} | {pressed}')

Inspecting GAME=1942-Nes-v0, EPISODE=0, rows=200
step | action_label | pressed_buttons
----------------------------------------------------------------------------------------------------
   0 | NOOP                                | 
   1 | B+RIGHT                             | B,RIGHT
   2 | B+LEFT                              | B,LEFT
   3 | A                                   | A
   4 | A                                   | A
   5 | UP                                  | UP
   6 | NOOP                                | 
   7 | B+LEFT                              | B,LEFT
   8 | A+DOWN                              | DOWN,A
   9 | A                                   | A
  10 | A+LEFT                              | LEFT,A
  11 | B+UP                                | B,UP
  12 | A+UP                                | UP,A
  13 | RIGHT                               | RIGHT
  14 | LEFT+START                          | START,LEFT
  15 | RIGHT+SELECT                        | SELECT,RIGHT
  16 